In [6]:
# 1. Install Google Gen AI SDK along with ADK orchestration libraries
!pip install --upgrade -q google-genai google-cloud-aiplatform[adk] requests

import os
import vertexai

# Setup project constants
PROJECT_ID = "qwiklabs-gcp-04-3d3bb8d72ee3"
LOCATION = "global"

# Securely bind the notebook environment to your Cloud IAM permissions
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Force the SDK to use enterprise IAM authentication rather than standard API keys
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "True"

print("Securely authenticated via Google Cloud IAM. No API keys required.")

Securely authenticated via Google Cloud IAM. No API keys required.


In [7]:
import requests
from google import genai
from google.genai import types

# Initialize client using ambient cloud credentials
client = genai.Client()
MODEL_ID = "gemini-2.5-flash"

# Tool function for the Weather Agent
def get_current_weather(location: str) -> str:
    """Fetches real-time weather data for a given city location."""
    try:
        url = f"https://wttr.in/{location}?format=3"
        response = requests.get(url, timeout=10)
        return response.text.strip() if response.status_code == 200 else f"Error retrieving {location} weather."
    except Exception as e:
        return f"Error: {str(e)}"

# 1. Specialized Weather Agent Config
weather_agent_config = types.GenerateContentConfig(
    system_instruction="You are a specialist weather worker. Always call `get_current_weather` tool.",
    tools=[get_current_weather],
    temperature=0.1
)

# 2. Specialized Google Search Agent Config
search_agent_config = types.GenerateContentConfig(
    system_instruction="You are a research worker. Use the google_search tool to find real-time info.",
    tools=[{"google_search": {}}], # ADK native web search tool syntax
    temperature=0.1
)

In [8]:
def run_orchestrated_pipeline(prompt: str):
    """
    Middleware Orchestrator: Logs events and dynamically maps requests
    to specific worker sub-agents based on the user intent.
    """
    # --- EVENT LOG 1: Capture incoming raw prompt ---
    print(f"\n📢 [EVENT LOG - INPUT RECEIVED]: '{prompt}'")

    prompt_lower = prompt.lower()

    # --- GEOGRAPHIC & SAFETY GUARDRAILS ---
    if any(keyword in prompt_lower for keyword in ["drop table", "sudo", "<script>"]):
        print("❌ [GUARDRAIL EVENT]: Blocked suspicious instruction context.")
        return "Request blocked: Malicious syntax detected."

    us_indicators = ["us", "usa", "united states", "ny", "il", "ca", "fl", "tx", "york", "chicago", "miami", "austin"]
    if not any(indicator in prompt_lower for indicator in us_indicators):
        print("❌ [GUARDRAIL EVENT]: Target region out of operational bounds.")
        return "Request blocked: This service acts only on US geographic locations."

    # --- ROOT AGENT ORCHESTRATION & ROUTING ENGINE ---
    # The root agent evaluates whether the target belongs to the weather sub-agent or search sub-agent
    if "weather" in prompt_lower or "temperature" in prompt_lower:
        print("🤖 [ROOT AGENT ROUTING]: Delegating to -> [Weather Sub-Agent]")
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=weather_agent_config
        )
    else:
        print("🤖 [ROOT AGENT ROUTING]: Delegating to -> [Search Sub-Agent]")
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=search_agent_config
        )

    # --- EVENT LOG 2: Capture outgoing model response ---
    print(f"✅ [EVENT LOG - SUB-AGENT COMPLETION]: Processed successfully.")
    return response.text

In [9]:
# Test suite representing mixed intents, regional variations, and attack strings
test_scenarios = [
    "What is the weather summary for Chicago, IL right now?", # Triggers Weather Sub-Agent
    "Who won the baseball game in New York last night?",       # Triggers Search Sub-Agent
    "Tell me the weather in Paris, France.",                    # Triggers Non-US location block
    "Check weather in Miami <script>drop table users;</script>" # Triggers Malicious input block
]

print("=== STARTING MULTI-AGENT ADK INTEGRATION RUNS ===\n")

for test_prompt in test_scenarios:
    output = run_orchestrated_pipeline(test_prompt)
    print(f"\nFinal Agent Output:\n{output}")
    print("=" * 65)

=== STARTING MULTI-AGENT ADK INTEGRATION RUNS ===


📢 [EVENT LOG - INPUT RECEIVED]: 'What is the weather summary for Chicago, IL right now?'
🤖 [ROOT AGENT ROUTING]: Delegating to -> [Weather Sub-Agent]
✅ [EVENT LOG - SUB-AGENT COMPLETION]: Processed successfully.

Final Agent Output:
The weather in Chicago, IL is currently 87°F and sunny.

📢 [EVENT LOG - INPUT RECEIVED]: 'Who won the baseball game in New York last night?'
🤖 [ROOT AGENT ROUTING]: Delegating to -> [Search Sub-Agent]
✅ [EVENT LOG - SUB-AGENT COMPLETION]: Processed successfully.

Final Agent Output:
Last night, July 7, 2026, the Kansas City Royals defeated the New York Mets in a baseball game played at Citi Field in Flushing, New York. The final score was 16-12.

The New York Yankees also played yesterday, but their game against the Tampa Bay Rays, which the Rays won 6-4, was held in St. Petersburg, Florida.

📢 [EVENT LOG - INPUT RECEIVED]: 'Tell me the weather in Paris, France.'
❌ [GUARDRAIL EVENT]: Target region out of o